In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [3]:
# Encode query
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

q2 = "How to isntall docker on windows?"
v2 = model.encode(q2)

# Encode document (one of the documents, just for an example)
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
# for cosin similarity we use dot only because the model normalizes the vectors. 
np.dot(v2,dv)

In [ ]:
# we take Q and compare it against all Doc: get top 5 (candidates)
from ingest import load_faq_data

documents = load_faq_data() # dict 
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [ ]:
# conv dict(documents) to text 
texts = []

for doc in documents: 
    text = doc['question'] + ' ' + doc["answer"]
    texts.append(text)

In [ ]:
# We split into batches because text is long, we embed each chunk then embed into vector space 

from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
vectors

In [ ]:
# turn into 2d array (matrix) where rows = doc/vectors and col = dim of vectors
X = np.array(vectors) # our documents 
X.shape

In [ ]:
cosine_sim(v1,X)
scores = X

In [ ]:
v1

In [ ]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [ ]:
v_query.shape

In [ ]:
X.shape

In [ ]:
scores = X.dot(v_query)

In [ ]:
scores

In [ ]:
# we select the closest want 
idx = np.argmax(scores) # idx of the largest 

In [ ]:
idx, scores[idx]